In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pds
from pprint import pprints
from datasets import load_dataset
import os
from dotenv import dotenv_values
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import PointStruct, Document
from groq import Groq

KeyboardInterrupt: 

In [ ]:
!pip install groq qdrant_client

In [ ]:
print("Hsel World")

In [4]:
# Load environment variables
config = dotenv_values(".env")
HF_TOKEN  = config.get("HF_TOKEN")
QDRANT_CLOUD_API_KEY = config.get("QDRANT_CLOUD_API_KEY")
QDRANT_CLOUD_ENDPOINT = config.get("QDRANT_CLOUD_ENDPOINT")
GROQ_API_KEY  = config["GROQ_API_KEY"]

# Make HuggingFace token available to the transformers library
os.environ["HF_TOKEN"] = HF_TOKEN

In [5]:
# Hugging Face Authentification
from huggingface_hub import login
login(token=HF_TOKEN)

In [6]:
#Embedding Model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
token = " هل يمكنني رؤية قائمة الطعام، من فضلك؟ "
vector = model.encode(token, normalize_embeddings=True)
len(vector)

384

In [8]:
collection_name = "Morrocan_Chat_Culture"
# Connection With QDrant Cloud
client_qdrant = QdrantClient(
    url=QDRANT_CLOUD_ENDPOINT,
    api_key=QDRANT_CLOUD_API_KEY,
    cloud_inference=True
)

In [28]:
# Collection Creation
client_qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config={
        "text": models.VectorParams(size=384, distance=models.Distance.COSINE), # we use cosin similarity
    },
    sparse_vectors_config={"text-sparse": models.SparseVectorParams()},
)

/tmp/ipykernel_12199/134971117.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client_qdrant.recreate_collection(


True

In [11]:
# client_qdrant.delete_collection(collection_name=collection_name)

In [12]:
# Example Of Insert Dense Vector Into Qdrant Cloud
client_qdrant.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=1,
            vector={
                "text": vector
            },
            payload={
                "chunk": token,
                "language": "ar"
            }

        )
    ]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [9]:
# Connection Wth Groq
client_groq = Groq(api_key=GROQ_API_KEY)

In [10]:
# Load Dataset From Hugging Face
dataset = load_dataset(
    "atlasia/Atlaset",
    split="train",
    streaming=True # we do not download this dataset we create object refrence o dataset
).select_columns(["text"])

In [11]:
# Get Emebddings for Chunks
def get_embeddings(sentences):
    embeddings = model.encode(sentences, normalize_embeddings=True)
    return embeddings

In [12]:
# Add Chunks to Vectorial DataBase
def add_sentences_to_Qdrant(embeddings, sentences, start_id):
    client_qdrant.upsert(collection_name=collection_name,
    points=[
        models.PointStruct(
            id = i + start_id,
            vector={
              "text": emb,  
            },
            payload={
                "token": sent
            }
        )
        for i, (emb, sent) in enumerate(zip(embeddings, sentences))
    ], wait=True)

In [14]:
# PIPELINE:
'''
    Read Batch
    Clean / Chunk
    Embed sentences
    UpSert Into Qdrants
    Discard from memory
    Next Bastch
'''

BATCH_SIZE = 10
total_tokens = 100000
counter = 0
sentences = []
start_id = 91800
for row in dataset:
    if total_tokens <= start_id:
        break

    sentences.append(row["text"])
    counter += 1
    if (BATCH_SIZE <= counter):
        # Get Embedding for this Batchsz
        embeddings = get_embeddings(sentences=sentences)
        embeddings = embeddings.tolist()
        
        # Push The sentences into QDrant VDB
        add_sentences_to_Qdrant(embeddings, sentences, start_id)
        sentences = []
        counter = 0
        start_id += BATCH_SIZE

print(f"Total points upserted: {start_id}")

Total points upserted: 100000


In [1]:
print("Hello World")

Hello World


In [1]:
# Check Similarity Between Question And Chunks
def get_relevent_chunks(embeddings_question):
    hits = client_qdrant.query_points(
        collection_name=collection_name,
        query=embeddings_question,
        using="text",
    )
    print(hits)

get_relevent_chunks("أكثر من نص مصاريف المخزن كانت كتمشي لبرا باش يخلصو الغرامات ديال الحرب و يشريو السلاح")

NameError: name 'model' is not defined

In [27]:
# Get hypothetical Embedding Documents

def get_llm_documents(question):
    """Generate a short hypothetical documentation passage for `question`."""
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
        {
            "role"   : "system",
            "content": (
            f"نتا مساعد كيعطي معلومات مفيدة. جاوب بالدارجة المغربية اللي ساهلة ومفهومة. "
            f"عطي جواب واضح ومختصر بلا إطالة. السؤال: {question}"
            ),
        }
        ],
        temperature=1,
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = [chunk.choices[0].delta.content for chunk in completion]
    res = [s for s in res if s]
    return "".join(res)

'يا سيدي، مغرب هوه بلاد ماشي كبيرة، ولكن فيها ماحقة من السياحة والفرص المالية. هي بلاد شعبية وودية ومتحضرين.\n\nمغرب تقدر تلاقي فيها:\n\n* سياحة على السواحل اللي هي مشهورين جدا\n* جو صحي وبيئة جميلة\n* ثقافة غنية وتراث قديم\n* أكلات الشعبية والمأكولات البحرية\n* فرص للعمل وتنمية اقتصادية\n\nمغرب شعب مازال فيه الكثير من الجود والود اللي فيه، وتعطيك كل خير.'

In [ ]:
# Get Reponse From LLM
def generate_response(context=context):
    pass

In [ ]:
def get_hyde_embedding(hyde_documents):
    # Make Documents With Chunks
    
    # Make Chunks With Embedding
    
    # Return Result
    pass

In [ ]:
def search_by_cosin(relevent_chunks, hyde_documents_chunks):
    pass

In [ ]:
# Define Question
question = " ة كاملة و ف 1985 ?"

# Get Question Embedding
embeddings_question = model.encode(question, normalize_embeddings=True).tolist()

# Get Relevent Chunks From Qdart
relevent_chunks = get_relevent_chunks(embeddings_question)

# Get hypothetical Embedding Documents
hyde_documents = get_llm_documents(question=question)

# Get Chunks from Hyde Documents
hyde_documents_chunks = get_hyde_embedding(hyde_documents)

# Filtre To Get Context With Cosin Similarity
filtred_documents = search_by_cosin(relevent_chunks, hyde_documents_chunks)
context = filtred_documents

# Get Response
response = generate_response(context=context)
print(response)

NameError: name 'model' is not defined

In [ ]:
# Hybrid Search

In [ ]:
# Hyde Architect

In [ ]:
# Get Relevent Documents

In [ ]:
# Get Context and Give LLM Context and Get Response